# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"], check=True)
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
else:
    hf_token = os.environ.get("HF_TOKEN")

import duckdb, pandas as pd, numpy as np
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE OR REPLACE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{hf_token}');")
print("DuckDB ready, HF secret registered.")

MONTH = "2026-03"
BASE = "hf://datasets/FlyRank/internship-warehouse"
print(f"Working month: {MONTH}")

DuckDB ready, HF secret registered.
Working month: 2026-03


In [11]:
raw = con.sql(f"""
    SELECT
        f.content_hash_id, f.client_hash_id, f.report_date,
        f.gsc_clicks, f.gsc_impressions, f.gsc_avg_position,
        c.content_type, c.word_count, c.content_created_date, c.is_deleted, c.is_published
    FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet') f
    JOIN read_parquet('{BASE}/dim_content.parquet') c ON f.content_hash_id = c.content_hash_id
    WHERE c.is_deleted = FALSE AND c.is_published = TRUE
""").df()

agg = raw.groupby("content_hash_id").agg(
    client_hash_id=("client_hash_id", "first"),
    gsc_clicks=("gsc_clicks", "sum"),
    gsc_impressions=("gsc_impressions", "sum"),
    gsc_avg_position=("gsc_avg_position", "mean"),
    content_type=("content_type", "first"),
    word_count=("word_count", "first"),
    content_created_date=("content_created_date", "first"),
).reset_index()
agg["ctr"] = (agg["gsc_clicks"] / agg["gsc_impressions"].replace(0, np.nan)).fillna(0).round(4)
agg["content_age_days"] = (pd.Timestamp(f"{MONTH}-01") - pd.to_datetime(agg["content_created_date"])).dt.days

agg_filtered = agg[agg["gsc_impressions"] >= 20].copy()
agg_filtered["position_tier"] = pd.cut(agg_filtered["gsc_avg_position"],
                                         bins=[0, 3, 10, 20, 1000],
                                         labels=["top_3", "page_1", "page_2_3", "deep"])
agg_filtered["tier_avg_ctr"] = agg_filtered.groupby("position_tier", observed=True)["ctr"].transform("mean")
agg_filtered["ctr_gap"] = agg_filtered["tier_avg_ctr"] - agg_filtered["ctr"]

# Baseline score, exactly as built in ML-07
agg_filtered["baseline_score"] = agg_filtered["ctr_gap"].clip(lower=0)

# Label, exactly as validated in ML-05 (median split on the same filtered data)
agg_filtered["label_low_ctr"] = (agg_filtered["ctr"] < agg_filtered["ctr"].median()).astype(int)

print(f"{len(agg_filtered):,} pages in working set")
print(f"Label balance: {agg_filtered['label_low_ctr'].value_counts(normalize=True).round(3).to_dict()}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

132,471 pages in working set
Label balance: {0: 0.501, 1: 0.499}


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*



My lane's question ("which pages first?") is a **ranking/scoring** problem per the
`training-honest-models` toolkit table — the right approach is a classifier's predicted
probability, evaluated at precision@K, not a raw classification accuracy.

Starting with **Logistic Regression** (readable, a clear coefficient per feature) then moving
to **Random Forest** (stronger, handles the nonlinear content_type/position interactions we
already found in ML-06). I'm skipping Gradient Boosting for this pass — per the skill,
"simplicity is a feature," and there's no evidence yet that the extra complexity is needed;
Random Forest already showed real signal in ML-05's leakage checklist without needing anything
heavier.

I'm also running **permutation importance** on the winning model, since the "what drives X"
row of the toolkit table specifically recommends this for interpretability — and because ML-06
already flagged that word_count and content_age tested FALSE as standalone signals, so if the
model leans heavily on them, that's a signal to investigate rather than celebrate (per the
skill's leakage sanity-check).

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*



**Grouped by `client_hash_id`**, not random or time-aware. This is the same design already
validated in ML-05's leakage checklist: pages from the same client can share hidden
characteristics (writing style, industry, template), so a random split risks letting the model
memorize per-client patterns and fake skill. A grouped split asks the honest question — does
this work on a client the model has never seen?

Not time-aware, because this is a **static, within-month opportunity ranking** (not a
future-decline prediction) — feature and label both come from the same March 2026 window by
design, which was already flagged as acceptable for this use case (and explicitly NOT
acceptable if reused as a future-prediction label) back in ML-04/ML-05.

In [12]:
from sklearn.model_selection import GroupKFold

feature_cols = ["gsc_avg_position", "gsc_impressions", "word_count", "content_age_days"]
X = agg_filtered[feature_cols].fillna(0)
y = agg_filtered["label_low_ctr"]
groups = agg_filtered["client_hash_id"]

gkf = GroupKFold(n_splits=5)
train_idx, test_idx = next(gkf.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Train: {len(X_train):,} pages | Test: {len(X_test):,} pages")
print(f"Client overlap between train/test: {len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))}")

Train: 105,977 pages | Test: 26,494 pages
Client overlap between train/test: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import numpy as np

np.random.seed(42)  # fixed seed, per the skill's reproducibility basics

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

logreg = LogisticRegression(random_state=42, max_iter=1000).fit(X_train, y_train)
rf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42).fit(X_train, y_train)

logreg_scores = logreg.predict_proba(X_test)[:, 1]
rf_scores = rf.predict_proba(X_test)[:, 1]
baseline_scores = agg_filtered.loc[X_test.index, "baseline_score"].values

base_rate = y_test.mean()
print(f"Base rate on test set: {base_rate:.3f}")

Base rate on test set: 0.466


In [14]:
results = []
for name, scores in [("baseline (ML-07 rule)", baseline_scores),
                       ("logistic regression", logreg_scores),
                       ("random forest", rf_scores)]:
    row = {"method": name}
    for k in (20, 50):
        row[f"precision@{k}"] = round(precision_at_k(scores, y_test.values, k), 3)
    results.append(row)

comparison = pd.DataFrame(results)
comparison["base_rate"] = round(base_rate, 3)
print(comparison.to_string(index=False))

               method  precision@20  precision@50  base_rate
baseline (ML-07 rule)           1.0           1.0      0.466
  logistic regression           1.0           1.0      0.466
        random forest           1.0           1.0      0.466


In [15]:
# Precision@20/50 all tied at 1.000 — test larger K to see if the methods actually differ
results_wide = []
for name, scores in [("baseline (ML-07 rule)", baseline_scores),
                       ("logistic regression", logreg_scores),
                       ("random forest", rf_scores)]:
    row = {"method": name}
    for k in (20, 50, 100, 200, 500):
        row[f"precision@{k}"] = round(precision_at_k(scores, y_test.values, k), 3)
    results_wide.append(row)

comparison_wide = pd.DataFrame(results_wide)
comparison_wide["base_rate"] = round(base_rate, 3)
print(comparison_wide.to_string(index=False))

               method  precision@20  precision@50  precision@100  precision@200  precision@500  base_rate
baseline (ML-07 rule)           1.0           1.0            1.0          1.000          1.000      0.466
  logistic regression           1.0           1.0            1.0          0.995          0.978      0.466
        random forest           1.0           1.0            1.0          0.995          0.984      0.466


**The real finding:** at small K (20, 50, 100) all three methods tie at a perfect 1.000 —
uninformative, since the test set is large enough that the very top of any reasonable ranking
is trivially correct. Widening to K=200 and K=500 reveals a real, consistent gap: the baseline
stays perfect (1.000) while both trained models slip slightly (0.995 → 0.978-0.984). The
baseline wins here, not by a landslide, but genuinely and consistently.

**Why the baseline wins:** this isn't surprising once you look at what each method actually
sees. The baseline's score is built directly from `ctr_gap`, which comes straight from the same
`ctr` column the label is thresholded from — a legitimate, transparent rule (not leakage), but
one with a structural home-field advantage at ordering extreme cases correctly. The trained
models never see `ctr` at all — they only see `gsc_avg_position`, `gsc_impressions`,
`word_count`, and `content_age_days`, and have to infer the pattern indirectly. That the models
get within 1-2 points of the baseline using only those indirect signals is itself a reasonable
result, just not a win.

**Honest verdict:** per the skill's own instruction not to reward complexity for its own sake —
neither trained model earns its place here. The baseline is simpler, fully transparent, and
performs at least as well (better at K=200/500) as either model. This is a legitimate outcome
of an honest comparison, not a failure of the exercise: sometimes the answer is "the baseline
still wins," and that's exactly the kind of result this internship is designed to surface
rather than hide.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [16]:
from sklearn.inspection import permutation_importance

# NOTE: run this against whichever model actually wins in the table above
best_model = rf  # change to logreg if logistic regression wins
perm = permutation_importance(best_model, X_test, y_test, n_repeats=10, random_state=42)
importances = pd.Series(perm.importances_mean, index=feature_cols).sort_values(ascending=False)
print("Permutation importance (best model):")
print(importances.round(4))

Permutation importance (best model):
gsc_impressions     0.2802
gsc_avg_position    0.0328
word_count          0.0014
content_age_days    0.0013
dtype: float64


In [17]:
test_df = agg_filtered.loc[X_test.index].copy()
test_df["pred_proba"] = rf_scores  # or logreg_scores, matching the best model
test_df["pred_label"] = (test_df["pred_proba"] >= 0.5).astype(int)
test_df["correct"] = (test_df["pred_label"] == test_df["label_low_ctr"])

error_by_tier = test_df.groupby("position_tier", observed=True)["correct"].agg(["mean", "count"]).round(3)
error_by_tier.columns = ["accuracy", "n"]
print("Accuracy by position tier (where is the model most wrong?):")
print(error_by_tier)

Accuracy by position tier (where is the model most wrong?):
               accuracy      n
position_tier                 
top_3             0.888   3631
page_1            0.835  13312
page_2_3          0.740   4253
deep              0.877   5295


In [18]:
wrong = test_df[~test_df["correct"]].sample(3, random_state=42)
print(wrong[["gsc_impressions", "gsc_avg_position", "position_tier", "ctr",
             "label_low_ctr", "pred_proba"]].to_string())

        gsc_impressions  gsc_avg_position position_tier     ctr  label_low_ctr  pred_proba
14614               283          4.734384        page_1  0.0071              0    0.573513
237845              985          0.913470         top_3  0.0000              1    0.257240
208400              117         17.672785      page_2_3  0.0085              0    0.809288


**What the model leans on:** `gsc_impressions` dominates permutation importance (0.2802),
roughly 8x larger than `gsc_avg_position` (0.0328), with `word_count` and `content_age_days`
contributing almost nothing (0.0014, 0.0013 respectively). This is consistent, not suspicious —
ML-06 already found both word_count and content_age tested FALSE as standalone CTR signals, so
their near-zero importance here confirms rather than contradicts that earlier finding. The
model is essentially learning "impression volume matters most for spotting the true zero-CTR
outliers," which connects directly to the measurement-anomaly pages flagged in ML-07's top-20
review (pages with tens of thousands of impressions and exactly zero clicks).

**Where it's most wrong:** accuracy is lowest in `page_2_3` (0.740, n=4,253) and highest in
`top_3` (0.888, n=3,631). This isn't simply a sample-size effect — `page_2_3` isn't the
smallest group. It's more likely that this tier sits in a genuinely messier zone: CTR
expectations here are less extreme than at `top_3` (very high-stakes, clear outliers) or `deep`
(uniformly low CTR, easy to predict as "low"), making the true label harder to separate from
noise in the middle of the distribution.

**Three wrong cases:**
- A `page_1` page with 283 impressions, position 4.7, CTR 0.71% — true label "not low" (0), but
  the model leaned toward "low" (0.57 confidence). This sits close to the median CTR threshold,
  an inherently ambiguous case near the label's own boundary, not a clear model failure.
- A `top_3` page with 985 impressions, position 0.9, CTR exactly 0.0000 — true label "low" (1),
  but the model was confident it wasn't (only 0.26). This is likely one of the
  measurement-anomaly pages already flagged in ML-07 (strong position, real traffic, zero
  clicks) — it looks like a top performer on every feature the model can see except the actual
  outcome, which is a genuinely hard case to catch without seeing CTR directly.
- A `page_2_3` page with only 117 impressions, position 17.7, CTR 0.85% — true label "not low"
  (0), but the model was fairly confident it was "low" (0.81). With such a small impression
  count, the CTR estimate itself is likely noisy and unstable, which plausibly explains why the
  model (relying on position and impressions) got misled here.

**Reproducibility:** random seed fixed at 42 throughout (train/test split, model
initialization, permutation importance) — rerunning this notebook reproduces the same
comparison table and numbers shown above.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.